# AERO EYES — chạy trên vast.ai (nhánh `test-geco2`, detector GeCo2)

Notebook này: cài dependency → clone code (nhánh `test-geco2`) → clone dataset từ Google Drive → chạy pipeline (`pipeline.detector: geco2`) → gộp + đánh giá kết quả.

**Lưu ý riêng cho vast.ai (khác Colab/Kaggle):**
- Không có `google.colab.drive.mount()` hay `/kaggle/input` tự động — dataset phải tự tải bằng `gdown`.
- Chọn template vast.ai có sẵn **CUDA toolkit đầy đủ** (không chỉ runtime) — bước build extension CUDA bên dưới cần `nvcc`. Image kiểu `pytorch/pytorch:*-devel` hoặc template có ghi "CUDA" của vast.ai là an toàn nhất.
- Instance vast.ai có thể bị xoá bất cứ lúc nào sau khi bạn dừng thuê — **tải kết quả về trước khi kết thúc** (cell cuối cùng có hướng dẫn).

**Phát hiện quan trọng khi chuẩn bị notebook này:** thư mục `GECO2/` trong repo hiện đang được git track như một **gitlink rỗng** (không có `.gitmodules` đăng ký) — nghĩa là `git clone` bình thường **sẽ ra thư mục `GECO2/` trống**, không có code bên trong. Notebook này đã tự xử lý (clone bù trực tiếp từ upstream `jerpelhan/GECO2` đúng commit đã ghim), nhưng đây là một lỗ hổng hạ tầng nên fix triệt để trong repo — nhắc bạn chủ động sửa (đăng ký submodule đúng cách hoặc bỏ nested `.git` để track như file thường).

## 0. Biến cấu hình chung — chỉnh ở đây trước khi chạy

In [ ]:
import os

# --- Repo chính (branch test-geco2) ---
REPO_URL = "https://github.com/Hoang-hai-yen/Test.git"
REPO_BRANCH = "test-geco2"
REPO_DIR = "/workspace/Test"

# --- GECO2 upstream (bù cho gitlink rỗng, xem cảnh báo phía trên) ---
GECO2_UPSTREAM_URL = "https://github.com/jerpelhan/GECO2.git"
GECO2_PINNED_COMMIT = "5b4fe9c4bc4a453bb366d314b80efb81450c51ef"

# --- GeCo2 pretrained weights (public HuggingFace asset, xem GECO2/README.md) ---
GECO2_WEIGHTS_URL = "https://huggingface.co/datasets/jerpelhan/geco2-assets/resolve/main/weights/CNTQG_multitrain_ca44.pth?download=true"

# --- Dataset trên Google Drive: 1 file .zip ---
# Lấy FILE_ID từ link share dạng https://drive.google.com/file/d/<FILE_ID>/view
GDRIVE_FILE_ID = "PASTE_HERE"  # <-- ACTION REQUIRED: dán File ID thật vào đây

# --- Chạy sample nào? None/rỗng = chạy hết mọi sample trong data_root ---
SAMPLE_ID = ""  # ví dụ "BlackBox_0" để chạy 1 sample; để rỗng "" = chạy tất cả

# --- Detector: "geco2" (mục tiêu notebook này) hoặc "legacy" để so sánh ---
DETECTOR = "geco2"

os.environ.update({
    "REPO_URL": REPO_URL, "REPO_BRANCH": REPO_BRANCH, "REPO_DIR": REPO_DIR,
    "GECO2_UPSTREAM_URL": GECO2_UPSTREAM_URL, "GECO2_PINNED_COMMIT": GECO2_PINNED_COMMIT,
    "GECO2_WEIGHTS_URL": GECO2_WEIGHTS_URL, "GDRIVE_FILE_ID": GDRIVE_FILE_ID,
    "SAMPLE_ID": SAMPLE_ID, "DETECTOR": DETECTOR,
})
print("OK — nhớ điền GDRIVE_FILE_ID thật trước khi chạy cell tải dataset.")

## 1. Kiểm tra GPU / CUDA toolkit

In [ ]:
!nvidia-smi
!echo "--- nvcc (cần cho bước build CUDA extension ở dưới) ---"
!nvcc --version || echo "CẢNH BÁO: không thấy nvcc — chọn lại template vast.ai có CUDA toolkit (devel), không phải bản runtime-only."

## 1a. Đồng bộ python giữa Jupyter kernel và shell (`%%bash`)

**Lỗi thường gặp trên vast.ai:** `%%bash`/`!pip install` chạy trong 1 shell subprocess có thể trỏ tới **python khác** với chính Jupyter kernel đang chạy notebook này (kernel thường nằm trong 1 venv riêng, còn shell mặc định vào python hệ thống) — cài package qua `%%bash` xong nhưng `import` trong cell Python thuần vẫn báo `ModuleNotFoundError`.

Cell dưới ưu tiên đúng thư mục chứa `sys.executable` (python của kernel) lên đầu `PATH`, để mọi `%%bash pip install` / `%%bash python -m ...` sau đó dùng cùng 1 môi trường với kernel. **Chạy cell này SỚM NHẤT (trước mọi bước cài đặt)** — nếu bạn đã lỡ chạy các cell cài đặt trước đó rồi mới thêm cell này, hãy **Restart Kernel rồi Run All lại từ đầu** để đảm bảo mọi thứ cài đúng chỗ.

In [ ]:
import os
import shutil
import sys

kernel_python = sys.executable
kernel_bin = os.path.dirname(kernel_python)
shell_python3 = shutil.which("python3")

print("Kernel python (sys.executable):", kernel_python)
print("Shell python3 (which python3) trước khi sửa:", shell_python3)

if shell_python3 and os.path.realpath(shell_python3) != os.path.realpath(kernel_python):
    print(">>> LỆCH MÔI TRƯỜNG -- ưu tiên thư mục của kernel python lên đầu PATH.")
else:
    print(">>> Khớp nhau (hoặc không xác định được shell python3) -- vẫn set PATH cho chắc.")

os.environ["PATH"] = kernel_bin + os.pathsep + os.environ.get("PATH", "")

print("Shell python3 sau khi sửa:", shutil.which("python3"))
print("pip sau khi sửa:", shutil.which("pip"))

## 1b. Đảm bảo lệnh `python` tồn tại

Nhiều image vast.ai (Debian/Ubuntu) chỉ có `python3`, không có `python` -- mọi lệnh `python ...` trong notebook này (build extension, `run_all`, `evaluate`, ...) sẽ báo `command not found` nếu bỏ qua bước này.

In [ ]:
%%bash
set -e
if ! command -v python &> /dev/null; then
    PY3=$(command -v python3)
    echo "Không có lệnh 'python' -- tạo symlink tới $PY3"
    ln -sf "$PY3" /usr/local/bin/python
fi
python --version
python -m pip --version

## 2. Clone repo

Clone repo chính (nhánh `test-geco2`). Nếu `GECO2/` trống sau khi clone (do gitlink chưa đăng ký — xem cảnh báo ở đầu notebook), tự clone bù từ upstream đúng commit đã ghim.

In [ ]:
%%bash
set -e
mkdir -p /workspace
rm -rf "$REPO_DIR"
git clone --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_DIR"
cd "$REPO_DIR"

if [ -z "$(ls -A GECO2 2>/dev/null)" ]; then
    echo ">>> GECO2/ rỗng (gitlink chưa đăng ký) — clone bù trực tiếp từ upstream ..."
    rm -rf GECO2
    git clone "$GECO2_UPSTREAM_URL" GECO2
    cd GECO2 && git checkout "$GECO2_PINNED_COMMIT" && cd ..
else
    echo ">>> GECO2/ đã có sẵn code (gitlink OK hoặc đã được fix trong repo)."
fi

echo "--- kiểm tra ---"
ls "$REPO_DIR"
ls "$REPO_DIR/GECO2" | head -5

## 3. Cài dependency của aero_eyes

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
pip install -q -r requirements.txt
pip install -q -e .
# Cần cho stage4.tracker=builtin (CSRT/KCF) -- opencv-python thường KHÔNG có cv2.legacy
pip install -q opencv-contrib-python-headless
echo "Done: aero_eyes deps"

## 4. Cài dependency riêng của GECO2

Theo `GECO2/install.sh`, bỏ qua `gradio`/`gradio_image_prompter` (chỉ cần cho demo UI, pipeline không dùng).

**Không** `pip install` package `sam2` -- code GECO2 tự import theo kiểu namespace package hai lớp (`sam2.sam2.modeling...`) dựa vào việc `GECO2/` (không phải `GECO2/sam2/`) nằm trên `sys.path`, việc pip-install `sam2` như 1 package độc lập sẽ làm sai đường import này.

**Cũng bỏ luôn `huggingface-hub==0.34.3`** mà `install.sh` gốc pin -- đã kiểm tra: package đó trong `install.sh` chỉ phục vụ `gradio`/`SAM2ImagePredictor.from_pretrained()` (không dùng ở đây), còn code GeCo2 thật sự chạy (`models/counter_infer.py`, `models/sam_mask.py`, backbone) không import `huggingface_hub` ở đâu cả. Giữ pin này sẽ xung đột với `transformers` (aero_eyes cần bản `huggingface-hub>=1.5,<2.0`) -- bỏ đi là an toàn, không mất chức năng gì.

In [ ]:
%%bash
set -e
pip install -q hydra-core omegaconf iopath scikit-image pycocotools einops
echo "Done: GECO2 deps (chưa pin numpy/pydantic -- xem cell pin version ở cuối)"

### (Tuỳ chọn) Fix `nvcc fatal: Unsupported gpu architecture 'compute_70'`

Chỉ chạy nếu bước build CUDA extension (mục 5) báo lỗi đúng như trên. Nguyên nhân: **`nvcc` của toolkit hệ thống trên image** đã lên CUDA 13.x và **bỏ hỗ trợ Volta/V100 (compute_70)**.

**ĐÃ XÁC NHẬN trên máy thật:** cài riêng `nvidia-cuda-nvcc-cu12` qua pip **không dùng được** — gói đó chỉ chứa `ptxas`/`nvvm` (thư viện backend), không có binary `nvcc` (compiler driver) thật. Cách đúng: cài hẳn **CUDA Toolkit 12.6 đầy đủ qua apt** (3 cell dưới: đặt cờ → apt install → hoàn tất env + đổi torch sang cu126). Tải khá nặng (~3-4GB), cần internet + quyền root trên instance (thường có sẵn trên vast.ai).

Nếu bước apt cũng lỗi (thiếu repo NVIDIA cho đúng bản OS, mạng chặn, v.v.): fallback chắc ăn nhất là **thuê lại instance vast.ai khác** với template ghi rõ CUDA ≤ 12.6 — đúng bản GECO2/install.sh đã test, không cần debug gì thêm.

In [ ]:
# Đặt True nếu build CUDA extension (mục 5) báo lỗi "nvcc fatal: Unsupported gpu architecture 'compute_70'".
FIX_NVCC_FOR_VOLTA = False

import os
import torch
print("torch hiện tại:", torch.__version__, "| cuda build:", torch.version.cuda,
      "| cuda available:", torch.cuda.is_available())

# Cell %%bash tiếp theo đọc cờ này qua biến môi trường.
os.environ["FIX_NVCC_FOR_VOLTA"] = "1" if FIX_NVCC_FOR_VOLTA else "0"

In [ ]:
%%bash
set -e
if [ "$FIX_NVCC_FOR_VOLTA" != "1" ]; then
    echo "FIX_NVCC_FOR_VOLTA=False -- bỏ qua cell này."
    exit 0
fi

. /etc/os-release
echo "OS phát hiện: $ID $VERSION_ID"
TAG="${ID}$(echo "$VERSION_ID" | tr -d '.')"
echo "Thử NVIDIA apt repo tag: $TAG"

KEYRING_URL="https://developer.download.nvidia.com/compute/cuda/repos/${TAG}/x86_64/cuda-keyring_1.1-1_all.deb"
if ! wget -q --spider "$KEYRING_URL"; then
    echo "Không có repo cho '$TAG', thử fallback 'ubuntu2404' ..."
    TAG="ubuntu2404"
    KEYRING_URL="https://developer.download.nvidia.com/compute/cuda/repos/${TAG}/x86_64/cuda-keyring_1.1-1_all.deb"
fi
echo "Dùng: $KEYRING_URL"

wget -q "$KEYRING_URL" -O /tmp/cuda-keyring.deb
dpkg -i /tmp/cuda-keyring.deb
apt-get update -qq
apt-get install -y -qq cuda-toolkit-12-6

echo "--- kiểm tra ---"
ls -d /usr/local/cuda-12.6
/usr/local/cuda-12.6/bin/nvcc --version

In [ ]:
if FIX_NVCC_FOR_VOLTA:
    import os, subprocess, sys

    cuda126_home = "/usr/local/cuda-12.6"
    nvcc126 = os.path.join(cuda126_home, "bin", "nvcc")
    if not os.path.exists(nvcc126):
        raise FileNotFoundError(
            f"{nvcc126} không tồn tại -- cell apt cài CUDA toolkit ở trên có thể đã lỗi, "
            "đọc lại log cell đó trước khi chạy tiếp."
        )

    os.environ["CUDA_HOME"] = cuda126_home
    os.environ["PATH"] = os.path.join(cuda126_home, "bin") + os.pathsep + os.environ["PATH"]

    # Torch cũng đổi sang build cu126 để khớp runtime lib với nvcc 12.6 vừa cài
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore",
                     "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
                     "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)

    print("--- kiểm tra lại ---")
    print("CUDA_HOME =", os.environ["CUDA_HOME"])
    subprocess.run(["nvcc", "--version"], check=True)

## 5. Build CUDA extension (Deformable-DETR ops) — bắt buộc cho GeCo2

`query_generator.py` của GECO2 cần `models.ops.modules.ms_deform_attn.MSDeformAttn`. Bước này build extension CUDA rồi copy source wrapper vào đúng vị trí `GECO2/models/ops/` (giống hệt `GECO2/install.sh`).

In [ ]:
%%bash
set -e
cd "$REPO_DIR/GECO2/Deformable-DETR/models/ops"
# --no-build-isolation: setup.py cần "import torch" ngay lúc build (torch.utils.cpp_extension) --
# build isolation mặc định của pip tạo 1 venv tạm KHÔNG có torch, gây "ModuleNotFoundError: No module named 'torch'".
CUDA_VISIBLE_DEVICES=0 python -m pip install --no-build-isolation .
cd "$REPO_DIR/GECO2"
rm -rf ./models/ops
cp -r ./Deformable-DETR/models/ops ./models/ops
# import torch TRƯỚC khi import extension -- extension .so cần libc10.so (nằm trong torch/lib/),
# chỉ định vị được nếu torch đã load trong cùng process trước đó (code thật của pipeline luôn
# import torch sớm nên không gặp lỗi này -- đây chỉ là yêu cầu của lệnh kiểm tra 1 dòng này).
python -c "import torch; import MultiScaleDeformableAttention; print('MultiScaleDeformableAttention import OK')"
echo "Done: CUDA ops extension built + copied"

## 6. Pin version cuối cùng (chạy SAU CÙNG trong phần cài đặt)

`GECO2/install.sh` cố tình đặt 2 dòng này cuối cùng — pip cài các gói khác ở trên có thể kéo numpy 2.x / pydantic mới hơn lên, 2 lệnh này ép lại đúng version GECO2 cần.

In [ ]:
%%bash
set -e
pip install -q "numpy<2"
pip install -q --force-reinstall "pydantic<2.11"
python -c "import numpy, pydantic; print('numpy', numpy.__version__, '| pydantic', pydantic.VERSION)"

## 7. Tải trọng số GeCo2

In [ ]:
%%bash
set -e
cd "$REPO_DIR/GECO2"
wget -q --show-progress -O CNTQG_multitrain_ca44.pth "$GECO2_WEIGHTS_URL"
ls -lh CNTQG_multitrain_ca44.pth

## 8. Tải dataset từ Google Drive (.zip)

**ACTION REQUIRED:** điền `GDRIVE_FILE_ID` thật ở Cell 0 trước khi chạy cell này.

In [ ]:
%%bash
set -e
if [ "$GDRIVE_FILE_ID" = "PASTE_HERE" ]; then
    echo "CHƯA điền GDRIVE_FILE_ID thật ở Cell 0 -- bỏ qua bước tải dataset."
    echo "Các cell setup ở trên vẫn dùng được để test riêng; quay lại đây khi có File ID."
else
    pip install -q -U gdown
    mkdir -p /workspace/aero_eyes_dataset_raw
    # gdown >=4.x bỏ flag --id -- truyền thẳng ID/URL làm positional argument.
    gdown "$GDRIVE_FILE_ID" -O /workspace/aero_eyes_dataset.zip
    unzip -q -o /workspace/aero_eyes_dataset.zip -d /workspace/aero_eyes_dataset_raw
    echo "--- cấu trúc sau khi giải nén (kiểm tra kỹ trước khi set DATA_ROOT/GT_FILE ở dưới) ---"
    find /workspace/aero_eyes_dataset_raw -maxdepth 4
fi

## 9. Trỏ đúng đường dẫn dataset

**Xem output `find` ở cell trên rồi chỉnh `DATA_ROOT`/`GT_FILE` cho khớp** — Drive/zip có thể có thêm 1 lớp thư mục con tuỳ cách bạn nén (giống lưu ý đã gặp thật với Kaggle trong `docs/COLAB_KAGGLE_GUIDE.md`), đừng đoán.

In [ ]:
import os

# <-- CHỈNH theo output `find` ở cell trên
DATA_ROOT = "/workspace/aero_eyes_dataset_raw"
GT_FILE = f"{DATA_ROOT}/annotations (1).json"
WORK_DIR = "/workspace/runs/exp001"

os.environ.update({"DATA_ROOT": DATA_ROOT, "GT_FILE": GT_FILE, "WORK_DIR": WORK_DIR})

samples = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))] if os.path.isdir(DATA_ROOT) else []
print(f"Tìm thấy {len(samples)} sample(s):", samples[:10])
print("GT file tồn tại:", os.path.exists(GT_FILE))

## 10. Smoke test — chạy 1 sample trước

Chạy thử 1 sample (dùng `SAMPLE_ID` đặt ở Cell 0, hoặc sample đầu tiên tìm được) để bắt lỗi sớm trước khi chạy hết dataset.

In [ ]:
%%bash
set -e
cd "$REPO_DIR"

SMOKE_SAMPLE="${SAMPLE_ID}"
if [ -z "$SMOKE_SAMPLE" ]; then
    SMOKE_SAMPLE=$(ls "$DATA_ROOT" | head -1)
fi
echo "Smoke test sample: $SMOKE_SAMPLE"

python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    --sample "$SMOKE_SAMPLE" \
    --set pipeline.detector="$DETECTOR" \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$GT_FILE" \
    --set project.work_dir="$WORK_DIR"

## 11. Chạy toàn bộ pipeline

Chạy hết mọi sample nếu `SAMPLE_ID` (Cell 0) để rỗng, hoặc chỉ 1 sample nếu đã set. Nếu bị 0 detection hàng loạt (domain gap ground→aerial), hạ `stage123_geco2.score_threshold_ratio` (xem phần Troubleshooting cuối notebook).

In [ ]:
%%bash
set -e
cd "$REPO_DIR"

SAMPLE_ARG=""
if [ -n "$SAMPLE_ID" ]; then
    SAMPLE_ARG="--sample $SAMPLE_ID"
fi

python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    $SAMPLE_ARG \
    --set pipeline.detector="$DETECTOR" \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$GT_FILE" \
    --set project.work_dir="$WORK_DIR"

## 12. Tổng hợp kết quả

`run_all` đã tự gộp mọi `submission.json` thành `WORK_DIR/submission_all.json` (trừ khi chạy `--no-merge`). Cell dưới chạy lại bước gộp tường minh (an toàn, upsert theo `video_id`, không tạo trùng) rồi in tóm tắt.

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
python -m scripts.merge_submissions \
    --config configs/config.yaml \
    --set data.data_root="$DATA_ROOT" \
    --set data.gt.global_file="$GT_FILE" \
    --set project.work_dir="$WORK_DIR"

In [ ]:
import json, os

merged_path = os.path.join(WORK_DIR, "submission_all.json")
data = json.load(open(merged_path, encoding="utf-8"))
print(f"submission_all.json: {len(data)} video(s)")
for e in data:
    n = len(e["annotations"][0]["bboxes"]) if e.get("annotations") else 0
    print(f"  {e['video_id']}: {n} frame(s) có box")

## 13. Đánh giá ST-IoU

In [ ]:
%%bash
set -e
cd "$REPO_DIR"
python -m aero_eyes.evaluate \
    --pred "$WORK_DIR/submission_all.json" \
    --gt "$GT_FILE" \
    --config configs/config.yaml

## 14. Đóng gói kết quả để tải về

vast.ai không tự lưu output như Kaggle's Output tab — **tải file này về trước khi dừng/huỷ instance**, qua file browser của Jupyter (chuột phải → Download) hoặc `scp -P <port> root@<host>:/workspace/aero_eyes_results.zip .` (lấy host/port từ nút Connect trên trang instance vast.ai).

In [ ]:
%%bash
set -e
cd /workspace
rm -f aero_eyes_results.zip
zip -q -r aero_eyes_results.zip \
    "$WORK_DIR/submission_all.json" \
    $(find "$WORK_DIR" -maxdepth 2 -name 'submission.json') \
    $(find "$WORK_DIR" -maxdepth 3 -path '*/viz/stage5/timeline.jpg')
ls -lh aero_eyes_results.zip
echo "Đã đóng gói: /workspace/aero_eyes_results.zip"

## Troubleshooting (đã gặp thật khi chạy dự án này trên các nền tảng khác, xem thêm `docs/COLAB_KAGGLE_GUIDE.md`)

| Lỗi | Nguyên nhân | Cách sửa |
|--|--|--|
| `bash: line N: python: command not found` | Image vast.ai chỉ có `python3`, không có `python` | Đã thêm Cell "1b" (tạo symlink) ngay sau bước kiểm tra GPU -- chạy cell đó trước mọi cell dùng lệnh `python` |
| `import torch` báo `ModuleNotFoundError` dù đã `pip install` xong | Kernel Jupyter chạy từ venv riêng (vd `/venv/main`), còn `%%bash`/`!pip` mặc định vào python hệ thống khác (`/usr/bin/python3`) — cài 1 nơi, kernel tìm 1 nơi khác | Đã thêm Cell "1a" (đồng bộ `PATH` theo `sys.executable`) ngay đầu notebook. Nếu gặp lỗi này: chạy cell "1a", rồi chạy LẠI toàn bộ cell cài đặt (mục 3, 4, 5) theo đúng thứ tự |
| `Getting requirements to build wheel ... ModuleNotFoundError: No module named 'torch'` (khi build ops, mục 5) | `pip install .` mặc định build trong venv cô lập tạm thời không có torch, trong khi `setup.py` cần `import torch` lúc build | Đã thêm `--no-build-isolation` vào lệnh `pip install .` ở mục 5 |
| `nvcc fatal: Unsupported gpu architecture 'compute_70'` (build ops, mục 5) | Toolkit CUDA hệ thống trên image là 13.x, đã bỏ hỗ trợ Volta/V100 (compute_70) — không phải lỗi version torch | Đặt `FIX_NVCC_FOR_VOLTA = True` ở 3 cell "Tuỳ chọn" trước mục 5 (cài hẳn CUDA Toolkit 12.6 qua apt — **không dùng cách pip `nvidia-cuda-nvcc-cu12`, đã xác nhận không hoạt động**), rồi chạy lại mục 5. Nếu apt cũng lỗi: thuê lại instance khác, chọn template CUDA ≤ 12.6 |
| `ImportError: libc10.so: cannot open shared object file` (ngay sau khi build ops thành công, mục 5) | Extension `.so` cần `libc10.so` (nằm trong `torch/lib/`), chỉ định vị được nếu `torch` đã import trong cùng process trước đó -- không phải lỗi build thật | Đã sửa lệnh kiểm tra ở mục 5 thành `python -c "import torch; import MultiScaleDeformableAttention; ..."` (import torch trước). Code pipeline thật không gặp lỗi này vì đã import torch từ sớm |
| `GECO2/` rỗng sau khi clone | gitlink chưa đăng ký `.gitmodules` trong repo | Notebook đã tự clone bù (Cell 2); nên fix triệt để trong repo sau |
| `nvcc: command not found` (không thấy nvcc luôn) | Template vast.ai không có CUDA toolkit nào cả (chỉ runtime) | Chọn lại instance/image có CUDA devel |
| `ModuleNotFoundError: MultiScaleDeformableAttention` | Bước 5 (build ops) chưa chạy hoặc chạy lỗi | Chạy lại toàn bộ Cell mục 5, đọc kỹ log lỗi build |
| `ImportError: sam2...` hoặc sai đường dẫn import | Lỡ `pip install` package `sam2` riêng (đừng làm việc này — xem ghi chú ở mục 4) | `pip uninstall sam2 SAM-2 -y` nếu đã trót cài |
| `transformers ... requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.34.3` | Cell mục 4 (bản cũ) pin `huggingface-hub==0.34.3` theo `install.sh` gốc — package đó GeCo2 không thật sự dùng, chỉ xung đột với `transformers` | Đã bỏ pin này khỏi Cell mục 4 hiện tại. Nếu đã lỡ chạy bản cũ: `pip install -q -U huggingface-hub` để trả lại bản `transformers` cần |
| `OpenCV tracker 'csrt' not found` (Stage 4) | Thiếu `cv2.legacy` | Đã cài `opencv-contrib-python-headless` ở mục 3 |
| Mọi sample đều 0 detection | GeCo2's score scale khác biệt theo video (domain gap ground→aerial) | Hạ `--set stage123_geco2.score_threshold_ratio=0.15` (mặc định 0.33) khi chạy lại Cell 11 |
| `data_root not found` | `DATA_ROOT` ở Cell 9 chưa khớp cấu trúc thật sau giải nén | Xem lại output `find` ở Cell 8, đừng đoán path |
| Mất hết kết quả sau khi đóng notebook | vast.ai xoá instance/volume khi bạn dừng thuê | Luôn chạy Cell 14 và tải `aero_eyes_results.zip` về trước khi dừng instance |